# ACS state-year WCF study: Colab fallback / from-scratch reproduction

This notebook reproduces the local `acs_state_year` study:

1. clones the WCF repository and installs dependencies,
2. downloads the Census ACS 1-year PUMS bulk files (`csv_pus.zip`, no API key),
3. builds K=25 state-year wage quantiles (age 16-64, WAGP>0) with 80-replicate SEs,
   converting wages to constant 2019 dollars with ADJINC and CPI-U NSA annual averages,
4. assembles the frozen adapter dataset (lagged X, ecdf moderator, pooled national q*),
5. runs the frozen WCF specification (M=10, 100 trees, lr=0.12, depth 4, 3 folds),
6. zips every output and downloads a single archive.

Defaults build outcome years 2013-2019 and the primary panel 2014-2019 (n=306).
Set `PANEL_START` / `PANEL_END` to change the estimation window; the build window
automatically includes `PANEL_START - 1` for the strictly lagged covariates. To
reproduce the extended 2010-2019 panel, set `BUILD_YEARS = list(range(2009, 2020))`
and `PANEL_START = 2010`.

Runtime: roughly 45-75 minutes on a standard Colab CPU runtime. Change
`SEEDS` to run more replications (adds ~3-8 min per seed).


In [ ]:
import os, subprocess, sys

REPO = "/content/wasserstein-causal-forests"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/hugogobato/wasserstein-causal-forests.git", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scipy", "scikit-learn", "pyarrow"], check=True)
sys.path.insert(0, os.path.join(REPO, "src"))
print("repo ready:", REPO)


In [ ]:
from pathlib import Path
import csv, json, time, urllib.request, zipfile

import numpy as np
import pandas as pd

PANEL_START, PANEL_END = 2014, 2019
SEEDS = [0]
# build window: the panel years plus one lag year
BUILD_YEARS = list(range(PANEL_START - 1, PANEL_END + 1))

K = 25
GRID = (np.arange(K, dtype=float) + 0.5) / K
N_REP = 80
REP_FACTOR = 4.0 / N_REP
BASE = ["ST", "AGEP", "WAGP", "PWGTP", "WKHP", "ADJINC", "SCHL", "SEX", "ESR"]
FEATURES = [
    "lag_median_wage_moderator", "lag_p10_wage", "log_real_min_wage",
    "lag_mw_above_fed", "log_wage_earners", "lag_share_female",
    "lag_share_ba_plus", "lag_share_age_25_54",
]

ROOT = Path("/content/wcf_acs")
PUMS = ROOT / "pums"
OUT = ROOT / "results"
YEARLY = OUT / "yearly"
for folder in (PUMS, YEARLY):
    folder.mkdir(parents=True, exist_ok=True)

CPI_PATH = ROOT / "cpi_u_nsa_monthly.csv"
if not CPI_PATH.exists():
    urllib.request.urlretrieve(
        "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCNS&cosd=2008-01-01&coed=2020-12-31",
        CPI_PATH,
    )
cpi = pd.read_csv(CPI_PATH)
cpi.columns = ["date", "cpi"]
cpi["year"] = pd.to_datetime(cpi["date"]).dt.year
CPI = cpi.groupby("year")["cpi"].mean()
print("CPI-U NSA annual averages ready; missing years:", sorted(set(BUILD_YEARS) - set(CPI.index)))


In [ ]:
def ensure_year(year):
    """Download and extract one ACS 1-year PUMS year; return its CSV parts."""
    zip_path = PUMS / f"csv_pus_{year}.zip"
    if not zip_path.exists():
        url = f"https://www2.census.gov/programs-surveys/acs/data/pums/{year}/1-Year/csv_pus.zip"
        print("downloading", url, flush=True)
        urllib.request.urlretrieve(url, zip_path)
    folder = PUMS / f"year_{year}"
    if not folder.exists():
        with zipfile.ZipFile(zip_path) as archive:
            members = [n for n in archive.namelist() if n.lower().endswith(".csv")]
            archive.extractall(folder, members=members)
    return sorted(folder.glob("*.csv"))

for year in BUILD_YEARS:
    parts = ensure_year(year)
    print(year, [p.name for p in parts])


In [ ]:
def read_header(path):
    with open(path, newline="") as handle:
        return next(csv.reader(handle))

def read_part(path):
    """Universe records of one PUMS part as a float32 block."""
    upper = {c.upper(): c for c in read_header(path)}
    weeks_col = "WKWN" if "WKWN" in upper else "WKW"
    columns = [upper[c] for c in BASE + [weeks_col]]
    columns += [upper[f"PWGTP{i}"] for i in range(1, N_REP + 1)]
    frame = pd.read_csv(path, usecols=columns, engine="pyarrow")
    for column in frame.columns:
        if not pd.api.types.is_numeric_dtype(frame[column]):
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame[
        (frame["ST"] >= 1) & (frame["ST"] <= 56)
        & (frame["AGEP"] >= 16) & (frame["AGEP"] <= 64)
        & (frame["WAGP"] > 0) & (frame["PWGTP"] > 0)
    ]
    keep = ["ST", "WAGP", "PWGTP", "AGEP", "WKHP", weeks_col, "SCHL", "SEX", "ESR", "ADJINC"]
    keep += list(columns[-N_REP:])
    return frame[keep].to_numpy(np.float32), weeks_col

def weighted_q(values_sorted, weights):
    cumulative = np.cumsum(weights)
    cumulative = cumulative / cumulative[-1]
    return np.interp(GRID, cumulative, values_sorted)

def quantiles_with_se(values, weights, replicate_weights):
    order = np.argsort(values, kind="mergesort")
    vs = values[order]
    q0 = weighted_q(vs, weights[order].astype(float))
    reps = np.empty((N_REP, K))
    for r in range(N_REP):
        wr = replicate_weights[order, r].astype(float)
        reps[r] = weighted_q(vs, wr) if wr.sum() > 0 else q0
    return q0, np.sqrt(REP_FACTOR * ((reps - q0) ** 2).sum(axis=0))

def build_year(year):
    parts = ensure_year(year)
    factor = float(CPI.loc[2019] / CPI.loc[year])
    chunks, weeks_col = {}, None
    for part in parts:
        block, weeks_col = read_part(part)
        states = block[:, 0].astype(int)
        for state in np.unique(states):
            chunks.setdefault(int(state), []).append(block[states == state])
    quantile_rows, summary_rows = [], []
    national_values, national_weights = [], []
    for state in sorted(chunks):
        block = np.vstack(chunks[state])
        wage = (block[:, 1] * block[:, 9] / 1e6 * factor).astype(float)
        weight = block[:, 2].astype(float)
        agep, wkhp, weeks = block[:, 3], block[:, 4], block[:, 5]
        schl, sex = block[:, 6], block[:, 7]
        q_wage, se_wage = quantiles_with_se(wage, weight, block[:, 10:])
        national_values.append(wage)
        national_weights.append(weight)
        ftyr = (wkhp >= 35) & ((weeks >= 50) if weeks_col == "WKWN" else ((weeks <= 2) & (weeks >= 1)))
        if ftyr.sum() > 0:
            q_ftyr, se_ftyr = quantiles_with_se(wage[fty], weight[fty], block[fty, 10:])
        else:
            q_ftyr = np.full(K, np.nan)
            se_ftyr = np.full(K, np.nan)
        for k in range(K):
            quantile_rows.append({
                "year": year, "state_fips": int(state), "k": k + 1, "u_k": float(GRID[k]),
                "q_wage": float(q_wage[k]), "q_wage_se": float(se_wage[k]),
                "q_ftyr": float(q_ftyr[k]), "q_ftyr_se": float(se_ftyr[k]),
            })
        summary_rows.append({
            "year": year, "state_fips": int(state), "n_records": int(block.shape[0]),
            "sum_pwgt": float(weight.sum()), "n_ftyr_records": int(ftyr.sum()),
            "sum_pwgt_ftyr": float(weight[ftyr].sum()),
            "median_wage": float(q_wage[12]), "p10_wage": float(q_wage[2]),
            "p90_wage": float(q_wage[22]), "median_ftyr": float(q_ftyr[12]) if ftyr.sum() else np.nan,
            "share_female": float(weight[sex == 2].sum() / weight.sum()),
            "share_ba_plus": float(weight[schl >= 21].sum() / weight.sum()),
            "share_age_25_54": float(weight[(agep >= 25) & (agep <= 54)].sum() / weight.sum()),
            "share_wagp_mult1000": float(np.mean(block[:, 1] % 1000 == 0)),
            "n_grid_se_zero": int(np.sum(se_wage <= 1e-9)),
            "cpi_factor_to_2019": factor,
        })
    pd.DataFrame(quantile_rows).to_csv(YEARLY / f"quantiles_long_{year}.csv", index=False)
    pd.DataFrame(summary_rows).to_csv(YEARLY / f"summary_{year}.csv", index=False)
    values = np.concatenate(national_values)
    weights = np.concatenate(national_weights)
    order = np.argsort(values, kind="mergesort")
    q_national = weighted_q(values[order], weights[order])
    pd.DataFrame({"u_k": GRID, "q_national": q_national}).to_csv(YEARLY / f"national_q_{year}.csv", index=False)
    np.savez_compressed(YEARLY / f"national_pool_{year}.npz",
                        wagp_real=values.astype(np.float32), pwgtp=weights.astype(np.float32))
    print("built year", year, "states:", len(summary_rows), flush=True)

for year in BUILD_YEARS:
    build_year(year)


In [ ]:
from wasserstein_causal_forests.applied.adapter import AppliedDataset, ecdf_moderator, weighted_quantiles

POLICY = Path(REPO) / "results/applied_study_exploration/state_year_pipeline/state_year_policy_panel_2010_2019.csv"
NORDIC = Path(REPO) / "results/applied_study_exploration/nordic_benchmark/nordic_qstar_k25_grid.csv"

summaries, quantiles = [], []
for year in BUILD_YEARS:
    s = pd.read_csv(YEARLY / f"summary_{year}.csv"); s["year"] = year; summaries.append(s)
    q = pd.read_csv(YEARLY / f"quantiles_long_{year}.csv"); q["year"] = year; quantiles.append(q)
panel = pd.concat(summaries, ignore_index=True)
qlong = pd.concat(quantiles, ignore_index=True)
wide_q = qlong.pivot(index=["year", "state_fips"], columns="k", values="q_wage")
wide_q.columns = [f"q_wage_{k}" for k in wide_q.columns]
wide_f = qlong.pivot(index=["year", "state_fips"], columns="k", values="q_ftyr")
wide_f.columns = [f"q_ftyr_{k}" for k in wide_f.columns]
panel = panel.merge(wide_q, left_on=["year", "state_fips"], right_index=True)
panel = panel.merge(wide_f, left_on=["year", "state_fips"], right_index=True)

policy = pd.read_csv(POLICY)
panel = panel.merge(policy[["statefips", "min_mw", "mw_above_fed", "medicaid_expanded"]],
                    left_on=["state_fips", "year"], right_on=["statefips", "year_only"],
                    how="left")
panel["min_mw_real"] = panel["min_mw"] * panel["cpi_factor_to_2019"]
panel["log_real_min_wage"] = np.log(panel["min_mw_real"])
panel["log_wage_earners"] = np.log(panel["sum_pwgt"])
panel = panel.sort_values(["state_fips", "year"]).reset_index(drop=True)
group = panel.groupby("state_fips", sort=False)
lag_map = {
    "lag_median_wage_moderator": "median_wage", "lag_p10_wage": "p10_wage",
    "lag_mw_above_fed": "mw_above_fed", "lag_min_mw_real": "min_mw_real",
    "lag_log_wage_earners": "log_wage_earners", "lag_share_female": "share_female",
    "lag_share_ba_plus": "share_ba_plus", "lag_share_age_25_54": "share_age_25_54",
}
for new, old in lag_map.items():
    panel[new] = group[old].shift(1)
panel = panel[(panel["year"] >= PANEL_START) & panel["lag_median_wage_moderator"].notna()].reset_index(drop=True)

Q = panel[[f"q_wage_{k}" for k in range(1, K + 1)]].to_numpy(float)
Q_ftyr = panel[[f"q_ftyr_{k}" for k in range(1, K + 1)]].to_numpy(float)
moderator_raw = panel["lag_median_wage_moderator"].to_numpy(float)
X = np.column_stack([
    ecdf_moderator(moderator_raw), panel["lag_p10_wage"], panel["log_real_min_wage"],
    panel["lag_mw_above_fed"], panel["lag_log_wage_earners"], panel["lag_share_female"],
    panel["lag_share_ba_plus"], panel["lag_share_age_25_54"],
])
A = panel["mw_above_fed"].to_numpy(np.int64)

values, weights = [], []
for year in range(PANEL_START, PANEL_END + 1):
    arrays = np.load(YEARLY / f"national_pool_{year}.npz")
    w = arrays["pwgtp"].astype(float)
    values.append(arrays["wagp_real"].astype(float))
    weights.append(w / w.sum())
q_star = weighted_quantiles(np.concatenate(values), np.concatenate(weights), levels=GRID)

meta = {
    "years": [PANEL_START, PANEL_END],
    "source": "Census ACS 1-year PUMS csv_pus.zip bulk files",
    "urls": {str(y): f"https://www2.census.gov/programs-surveys/acs/data/pums/{y}/1-Year/csv_pus.zip" for y in BUILD_YEARS},
    "universe": "AGEP 16-64, WAGP>0, PWGTP>0, ST 1..56",
    "deflator": "CPI-U NSA annual averages (FRED CPIAUCNS) to 2019 dollars",
    "q_star_primary": "equal-year-weight pooled national quantiles",
    "features": FEATURES,
}
ds = AppliedDataset(study=f"acs_state_year_colab_{PANEL_START}_{PANEL_END}", X=X, A=A, Q=Q,
                    moderator_raw=moderator_raw, q_star=q_star, feature_names=FEATURES, meta=meta)
ds.save(OUT / "data")
ds_ftyr = AppliedDataset(study=ds.study + "_ftyr", X=X, A=A, Q=Q_ftyr, moderator_raw=moderator_raw,
                         q_star=q_star, feature_names=FEATURES, meta=meta)
ds_ftyr.save(OUT / "data_ftyr")
np.savez_compressed(OUT / "treatments.npz",
                    A_primary=A,
                    A_medicaid=panel["medicaid_expanded"].to_numpy(np.int64),
                    A_real_increase=(panel["min_mw_real"].to_numpy(float) > panel["lag_min_mw_real"].to_numpy(float) + 1e-9).astype(np.int64),
                    A_lag=panel["lag_mw_above_fed"].to_numpy(np.int64))
panel.to_csv(OUT / "panel.csv", index=False)
print("dataset n=", ds.X.shape[0], "treated=", int(A.sum()), "q_star median=", float(q_star[12]))


In [ ]:
from wasserstein_causal_forests.applied.adapter import run_wcf, run_placebo

weights_grid = np.full(K, 1.0 / K)
mean_star = float(q_star @ weights_grid)

def h_ref_norm(q):
    block = np.asarray(q, dtype=float)
    centre = block @ weights_grid
    return np.sqrt(((block / centre[:, None] - q_star[None, :] / mean_star) ** 2) @ weights_grid)

extras = {
    "p10": lambda q: np.asarray(q, float)[:, 2],
    "p50": lambda q: np.asarray(q, float)[:, 12],
    "p90": lambda q: np.asarray(q, float)[:, 22],
    "ref_norm": h_ref_norm,
}
runs = {}
for seed in SEEDS:
    model, results = run_wcf(ds, random_state=seed, extra_functionals=extras)
    runs[f"primary_seed{seed}"] = results
    print("seed", seed, "selected shrinkage", results["selected_contrast_shrinkage"], flush=True)
for seed in SEEDS:
    runs[f"placebo_seed{seed}"] = run_placebo(ds, seeds=(seed,), extra_functionals=extras)[0]
with open(OUT / "wcf_runs.json", "w") as handle:
    json.dump(runs, handle, indent=2, default=str)
for key, run in runs.items():
    print(key, {name: round(run["marginal_dr"][name], 2) for name in ["mean", "sd", "p10", "p50", "p90"]})


In [ ]:
import shutil

archive = shutil.make_archive("/content/wcf_acs_state_year_outputs", "zip", ROOT)
print("archive:", archive, os.path.getsize(archive) / 1e6, "MB")
try:
    from google.colab import files
    files.download(archive)
    print("Downloaded:", archive)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
